# Session 7 — Convolutional Neural Networks

**Sebastián Ríos Saucedo · Computer Vision & Speech Recognition · EADA**

**Question:** the network now sees the pixels. Does that beat the handcrafted numbers?

**The plan** (class notebook, adapted to my repo and my Milestone 1 split)
1. Convolution, ReLU and pooling: first with small numbers we can check by hand.
2. A small CNN from scratch, tracing the shape after every layer.
3. A pretrained ResNet-18: frozen, then fine-tuned (`layer4`).
4. **Mini challenge (slide 29):** unfreeze `layer3 + layer4 + fc` and compare with `layer4` only.
5. **Homework (slide 30):** optional full fine-tuning, then the test set (once) and the final scoreboard of Sessions 5–7, plus the EfficientNet / DenseNet reading.

**How to run:** VS Code → select the `venv` kernel → *Restart* → *Run All*. Everything this notebook produces goes to `outputs/session_07/` and `figures/session7_*.png`. Afterwards, `python scripts/13_session7_reports.py` builds the two PDFs for the Aula from those files.

### 0. Setup

#### 0.1 Libraries, data location and device

Same as the class, but the repo root is found through `milk10k_pipeline/config.py` (my repo has no `common/` folder), exactly as in my Session 6 notebook.

In [ ]:
import os
import sys
import time
import json
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# New libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image

# Find the repo root by walking up until we see milk10k_pipeline/config.py
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "milk10k_pipeline" / "config.py").is_file())
sys.path.insert(0, str(ROOT))

from milk10k_pipeline import config     # data only: DATA_DIR, IMAGE_DIR, image_path(), SPLIT_DIR

OUT = config.OUTPUT_DIR / "session_07"  # every table / JSON this notebook writes goes here
OUT.mkdir(parents=True, exist_ok=True)
FIG = config.FIGURE_DIR                 # figures go to figures/session7_*.png
FIG.mkdir(parents=True, exist_ok=True)

# The 0.8 GB image cache does NOT go inside the repo (git) nor OneDrive: it goes to a local cache folder
CACHE_DIR = Path(os.environ.get("MILK10K_CACHE_DIR", Path.home() / ".cache" / "milk10k"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Use the GPU if there is one, otherwise the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.__version__, "| threads:", torch.get_num_threads())
print(config.describe())

# Fix the random seed so that results can be repeated
torch.manual_seed(0)

# Session 7 results are collected here and saved to outputs/session_07/results.json after every model
RESULTS = {}
def save_results():
    (OUT / "results.json").write_text(json.dumps(RESULTS, indent=2))

### 1. Convolution by hand → `nn.Conv2d`

A convolutional layer does the same slide-multiply-sum as Session 4. The difference is that its kernel numbers are learned.

#### 1.1 Convolution with PyTorch, using Session 4's kernel

PyTorch expects images as **(batch, channels, height, width)**, so our one gray image has shape (1, 1, 5, 5).

In [ ]:
# The toy lesion: 100 = skin, 10 = lesion
toy = torch.tensor([[100, 100, 100, 100, 100],
                    [100, 100, 100, 100, 100],
                    [100, 100,  10,  10,  10],
                    [100, 100,  10,  10,  10],
                    [100, 100,  10,  10,  10]], dtype=torch.float32)

# Sobel x: right column minus left column, so it reacts to vertical edges
sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]])

# No padding and stride 1, so 5x5 in -> 3x3 out
out = F.conv2d(toy.view(1, 1, 5, 5), sobel_x.view(1, 1, 3, 3))    # (batch, channels, H, W)
print(out.shape)
out[0, 0]          # [[-90, -90, 0], [-270, -270, 0], [-360, -360, 0]]

#### 1.2 The same operation as a layer

`nn.Conv2d` creates the kernel with **random** numbers. Here we overwrite them with Sobel, only to show that it is the same operation.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
print(conv.weight.shape, conv.bias.shape)             # torch.Size([1, 1, 3, 3]) torch.Size([1])
print(conv.weight.data[0, 0])                         # random numbers: what a fresh CNN starts with

with torch.no_grad():                                 # hand-design it, Session 4 style
    conv.weight[:] = sobel_x.view(1, 1, 3, 3)
    conv.bias.zero_()
conv(toy.view(1, 1, 5, 5))[0, 0]                      # same -90 / -270 / -360 as above

#### 1.3 The parameter explosion: dense vs. convolution

In [ ]:
count = lambda m: sum(p.numel() for p in m.parameters())   # total number of weights of a layer / model

dense = nn.Linear(224 * 224 * 3, 128)                 # 150,528 inputs x 128 neurons + 128 biases
conv32 = nn.Conv2d(3, 32, kernel_size=3, padding=1)   # 3x3x3 weights x 32 filters + 32 biases
print(f"dense: {count(dense):,}   conv: {count(conv32):,}   ratio: {count(dense) / count(conv32):,.0f}x")
# dense: 19,267,712   conv: 896   ratio: 21,504x

#### 1.4 What comes out of each layer

In [ ]:
x = torch.randn(1, 3, 224, 224)                # one fake RGB image
print(conv32(x).shape)                         # torch.Size([1, 32, 224, 224]) -> 32 feature maps
print(dense(x.flatten(1)).shape)               # torch.Size([1, 128])          -> spatial layout gone
del dense                                      # 19 M weights we don't need any more

### 2. ReLU and max pooling by hand

#### 2.1 ReLU, then max pooling

In [ ]:
fmap = torch.tensor([[-3.,  5.,  2.,  0.],
                     [ 1.,  8., -1.,  4.],
                     [ 0., -2.,  6.,  7.],
                     [ 3.,  1., -4.,  2.]])
relu_map = F.relu(fmap)                        # negatives -> 0
pooled = F.max_pool2d(relu_map.view(1, 1, 4, 4), kernel_size=2)   # 4x4 -> 2x2
print(relu_map)
print(pooled[0, 0])                            # [[8, 4], [3, 7]]

#### 2.2 Small shifts don't change the result

In [ ]:
shifted = relu_map.clone()
shifted[1, 1], shifted[1, 0] = 0., 8.           # the 8 moves one pixel left
print(shifted)
print(F.max_pool2d(shifted.view(1, 1, 4, 4), 2)[0, 0])   # still [[8, 4], [3, 7]]

#### 2.3 The output-size formula: **out = ⌊(n + 2·pad − kernel) / stride⌋ + 1**

In [ ]:
def out_size(n, kernel, stride=1, pad=0):
    return (n + 2 * pad - kernel) // stride + 1

print(out_size(224, kernel=3, pad=1))            # 224: 'same' padding
print(out_size(224, kernel=2, stride=2))         # 112: pooling halves

### 3. A small CNN from scratch — trace every shape

#### 3.1 Defining the network

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        # Feature extractor. Padding 1 keeps the size; each MaxPool2d(2) halves it.
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # block 1
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # block 2
        )
        # Classifier: 32 maps of 56x56 -> a list of 100,352 numbers -> 3 class scores
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 56 * 56, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

#### 3.2 Checking every shape before training

In [ ]:
model = SimpleCNN()
x = torch.zeros(1, 3, 224, 224)                  # one empty image: only its shape matters here
for layer in [*model.features, *model.classifier]:
    x = layer(x)
    print(f"{layer.__class__.__name__:10s} {str(tuple(x.shape)):20s} params: {count(layer):>8,}")
print(f"total parameters: {count(model):,}")     # 306,147 (98% in the last Linear)

### What would happen if the input size is different than 224×224?

The conv and pool layers don't care: they slide over any size, so a 256×256 image just gives 32 maps of 64×64. The problem is the **Linear** layer: it was built for exactly 32·56·56 = 100,352 inputs, and 32·64·64 = 131,072 numbers don't fit, so PyTorch raises a shape error. The cell below shows it, and shows the fix modern CNNs (ResNet included) use: **global average pooling** (`nn.AdaptiveAvgPool2d(1)`) turns every map into one number, so the head always receives 32 numbers whatever the image size, and the head shrinks from 301,059 to 99 weights.

In [ ]:
try:
    SimpleCNN()(torch.zeros(1, 3, 256, 256))
except RuntimeError as e:
    print("256x256 ->", str(e).splitlines()[0])

gap_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 3))   # global average pooling head
feats = SimpleCNN().features
for size in (224, 256, 300):
    print(size, "->", tuple(gap_head(feats(torch.zeros(1, 3, size, size))).shape), "| head weights:", count(gap_head))

### 4. MILK10k images — my split, then a one-time cache

#### 4.1 Split
The class used `milk10k.make_splits(seed=0)`. I use **my Milestone 1 split** (lesion-level StratifiedGroupKFold, seed 42), the same one I used in Sessions 5 and 6, keeping one dermoscopic image per lesion. So all scores in the scoreboard are comparable with each other (not with the class numbers, which use a different split: 3,593 / 599 / 1,048 lesions).

In [ ]:
import torchvision.transforms as T
from torch.utils.data import DataLoader

train_df, val_df, test_df = [pd.read_csv(config.SPLIT_DIR / f"{s}.csv") for s in ("train", "val", "test")]
train_df, val_df, test_df = [d[d["image_type"] == "dermoscopic"].reset_index(drop=True)
                             for d in (train_df, val_df, test_df)]        # one image per lesion
for d in (train_df, val_df, test_df):
    assert d["lesion_id"].is_unique                                       # really one image per lesion
assert not set(train_df.lesion_id) & (set(val_df.lesion_id) | set(test_df.lesion_id))   # no leakage

CLASSES = ["Benign", "Indeterminate", "Malignant"]            # same order as my label_map.json (y_primary)
to_int = {c: i for i, c in enumerate(CLASSES)}
print(len(train_df), len(val_df), len(test_df))               # 3742 749 749

#### 4.2 Decode each image once and keep it

Same idea as the class: open + resize every JPEG once, keep everything as one `uint8` tensor (≈0.8 GB) and save it. The file goes to a local cache folder outside the repo and outside OneDrive, so git and OneDrive never try to upload it.

In [ ]:
cache = CACHE_DIR / f"images_224_seed{config.SEED}.pt"
if cache.exists():
    data = torch.load(cache, weights_only=True)
else:
    t0 = time.perf_counter()
    data = {}
    for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
        X = torch.empty((len(split), 3, 224, 224), dtype=torch.uint8)    # uint8: 4x less memory than float
        for i, isic_id in enumerate(split["isic_id"]):
            img = Image.open(config.image_path(isic_id)).convert("RGB").resize((224, 224))
            X[i] = torch.from_numpy(np.array(img)).permute(2, 0, 1)     # (H, W, 3) -> (3, H, W)
        data[f"X_{name}"] = X
        data[f"y_{name}"] = torch.tensor(split["diagnosis_1"].map(to_int).to_numpy())
    torch.save(data, cache)
    print(f"cache built in {time.perf_counter() - t0:.0f} s -> {cache}")

X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]
print(X_train.shape, X_train.dtype, torch.bincount(y_train))    # my split: Benign 1058, Indeterminate 88, Malignant 2596

#### 4.3 A quick look at the data

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, i in zip(axes, range(6)):
    ax.imshow(X_train[i].permute(1, 2, 0)); ax.set_title(CLASSES[y_train[i]]); ax.axis("off")

### 5. The training toolkit — normalize, augment, fit, evaluate

#### 5.1 `prep`: normalize like ImageNet

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)   # ImageNet statistics
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def prep(xb):
    return (xb.to(device).float() / 255 - MEAN) / STD

print(((200 / 255) - 0.485) / 0.229)             # 1.31

#### 5.2 `augment`: random flips (a lesion has no 'up')

In [ ]:
def augment(xb):
    n = xb.size(0)
    flip_h = torch.rand(n, 1, 1, 1, device=xb.device) < 0.5  # one coin per image
    xb = torch.where(flip_h, xb.flip(3), xb)                 # left-right
    flip_v = torch.rand(n, 1, 1, 1, device=xb.device) < 0.5
    return torch.where(flip_v, xb.flip(2), xb)               # up-down

#### 5.3 `predict` and `fit`: Session 6's training loop

Same loop as the class. **My only change:** `fit` also stores the **training seconds** of every epoch (`train_s`, without the validation pass) and the total epoch time (`epoch_s`), because the mini challenge asks for seconds per epoch. `record()` saves each model's numbers to `results.json` for the report.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Class weights, as in Session 6: n / (3 * n_c), so mistakes on rare classes cost more
class_weights = (len(y_train) / (3 * torch.bincount(y_train).float())).to(device)
print(class_weights)

@torch.no_grad()
def predict(model, X, batch_size=128):
    model.eval()
    return torch.cat([model(prep(X[i:i + batch_size])).argmax(1).cpu() for i in range(0, len(X), batch_size)])

def fit(model, params, epochs, lr, batch_size=64, aug=True):
    model.to(device)
    optimizer = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    history, best = [], {"f1": -1}
    for epoch in range(epochs):
        model.train()
        t0, total = time.perf_counter(), 0.0
        order = torch.randperm(len(X_train))
        for i in range(0, len(X_train), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = prep(X_train[idx]), y_train[idx].to(device)
            if aug:
                xb = augment(xb)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        train_s = time.perf_counter() - t0                              # NEW: training time only
        f1 = f1_score(y_val, predict(model, X_val), average="macro")
        epoch_s = time.perf_counter() - t0                              # NEW: training + validation
        history.append({"epoch": epoch, "train_loss": total / len(X_train), "val_f1": f1,
                        "train_s": train_s, "epoch_s": epoch_s})
        print(f"epoch {epoch}  loss {total / len(X_train):.3f}  val macro-F1 {f1:.3f}  "
              f"(train {train_s:.0f} s, total {epoch_s:.0f} s)")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
    model.load_state_dict(best["state"])          # early stopping: keep the best epoch
    return pd.DataFrame(history).set_index("epoch"), best

def record(key, label, model, hist, best, n_trainable, lr, extra=None):
    """Save the numbers of one trained model (validation only; test is opened in section 12)."""
    RESULTS[key] = {"label": label, "trainable_params": int(n_trainable), "total_params": int(count(model)),
                    "lr": lr, "epochs": int(len(hist)),
                    "sec_per_epoch_train": round(float(hist["train_s"].mean()), 1),
                    "sec_per_epoch_total": round(float(hist["epoch_s"].mean()), 1),
                    "best_val_f1": round(float(best["f1"]), 4), "best_epoch": int(best["epoch"]),
                    "val_f1_per_epoch": [round(float(v), 4) for v in hist["val_f1"]],
                    "train_loss_per_epoch": [round(float(v), 4) for v in hist["train_loss"]],
                    "val_pred": predict(model, X_val).tolist(), **(extra or {})}
    save_results()

### 6. Train the CNN from scratch

Random weights, 3,742 lesions, 10 epochs.

In [ ]:
torch.manual_seed(0)
scratch = SimpleCNN()
hist_scratch, best_scratch = fit(scratch, scratch.parameters(), epochs=10, lr=1e-3)
record("scratch", "SimpleCNN from scratch (2 conv blocks)", scratch, hist_scratch, best_scratch,
       count(scratch), lr=1e-3)
print("best epoch", best_scratch["epoch"], "val macro-F1", round(best_scratch["f1"], 3))

#### 6.2 What did its first layer learn?

In [ ]:
w = scratch.features[0].weight.detach().cpu()                       # (16, 3, 3, 3)
fig, axes = plt.subplots(1, 16, figsize=(16, 1.4))
for ax, f in zip(axes, w):
    f = f.permute(1, 2, 0); ax.imshow((f - f.min()) / (f.max() - f.min())); ax.axis("off")

### 7. A pretrained ResNet-18

#### 7.1 Loading the network

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)       # downloads ~45 MB the first time
print(resnet.fc)                                                # Linear(in_features=512, out_features=1000)
print(f"{count(resnet):,} parameters")                          # 11,689,512
for name, child in resnet.named_children():
    print(f"{name:8s} {count(child):>10,}")

#### 7.2 Its first layer: 64 filters learned on ImageNet

In [ ]:
w = resnet.conv1.weight.detach()                                 # (64, 3, 7, 7)
fig, axes = plt.subplots(4, 16, figsize=(14, 3.6))
for ax, f in zip(axes.ravel(), w):
    f = f.permute(1, 2, 0); ax.imshow((f - f.min()) / (f.max() - f.min())); ax.axis("off")

#### 7.3 One lesion through the network

In [ ]:
x = prep(X_val[:1])
resnet.to(device).eval()
with torch.no_grad():
    h = resnet.maxpool(resnet.relu(resnet.bn1(resnet.conv1(x))))
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
    axes[0].imshow(X_val[0].permute(1, 2, 0)); axes[0].set_title(CLASSES[y_val[0]]); axes[0].axis("off")
    for ax, name in zip(axes[1:], ["layer1", "layer2", "layer3", "layer4"]):
        h = getattr(resnet, name)(h)
        k = h[0].flatten(1).std(1).argmax()                      # the map with the largest variation
        ax.imshow(h[0, k].cpu(), cmap="magma"); ax.set_title(f"{name} {tuple(h.shape[1:])}"); ax.axis("off")

### 8. Feature extraction — 512 learned numbers instead of our 33

**Strategy 1: freeze the backbone.**

#### 8.1 Turning each lesion into 512 numbers

In [ ]:
resnet.fc = nn.Identity()
resnet.eval()

@torch.no_grad()
def embed(X, batch_size=128):
    return torch.cat([resnet(prep(X[i:i + batch_size])).cpu() for i in range(0, len(X), batch_size)])

t0 = time.perf_counter()
F_train, F_val, F_test = embed(X_train), embed(X_val), embed(X_test)
embed_s = time.perf_counter() - t0
print(F_train.shape, f"{embed_s:.0f} s")       # torch.Size([3742, 512])

#### 8.2 Training only a new last layer (number to beat on my split: my Session 5 baseline 0.545 / forest 0.534)

In [ ]:
from torch.utils.data import TensorDataset

torch.manual_seed(0)
head = nn.Linear(512, 3).to(device)                       # 512 x 3 weights + 3 biases = 1,539
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
loader = DataLoader(TensorDataset(F_train, y_train), batch_size=64, shuffle=True)
head_hist = []
for epoch in range(25):
    t0 = time.perf_counter()
    for fb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(head(fb.to(device)), yb.to(device))
        loss.backward()
        optimizer.step()
    train_s = time.perf_counter() - t0
    with torch.no_grad():
        f1 = f1_score(y_val, head(F_val.to(device)).argmax(1).cpu(), average="macro")
    head_hist.append({"epoch": epoch, "train_loss": loss.item(), "val_f1": f1,
                      "train_s": train_s, "epoch_s": time.perf_counter() - t0})
head_hist = pd.DataFrame(head_hist).set_index("epoch")

with torch.no_grad():
    head_val_pred = head(F_val.to(device)).argmax(1).cpu()
head_val_f1 = f1_score(y_val, head_val_pred, average="macro")
print(f"frozen ResNet-18 + linear head: val macro-F1 = {head_val_f1:.3f}")

# As in class, the head keeps its LAST epoch (no early stopping), so "best" = last epoch here
RESULTS["frozen"] = {"label": "ResNet-18 frozen + linear head (feature extraction)",
                     "trainable_params": int(count(head)), "total_params": int(count(resnet) + count(head)),
                     "lr": 1e-3, "epochs": 25,
                     "sec_per_epoch_train": round(float(head_hist["train_s"].mean()), 2),
                     "sec_per_epoch_total": round(float(head_hist["epoch_s"].mean()), 2),
                     "embedding_seconds_once": round(embed_s, 1),
                     "best_val_f1": round(float(head_val_f1), 4), "best_epoch": 24,
                     "val_f1_per_epoch": [round(float(v), 4) for v in head_hist["val_f1"]],
                     "val_pred": head_val_pred.tolist()}
save_results()

#### 8.3 Where are the errors?

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, head_val_pred, display_labels=CLASSES, cmap="Oranges");

### 9. Partial fine-tuning — unfreeze `layer4`

**Strategy 2: let the last block adapt to skin lesions.**

#### 9.1 Choosing what to train

In [ ]:
torch.manual_seed(0)
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 3)                                     # new head: 3 classes
for name, p in model.named_parameters():
    p.requires_grad = name.startswith("layer4") or name.startswith("fc")

trainable = [p for p in model.parameters() if p.requires_grad]
n_trainable_l4 = sum(p.numel() for p in trainable)
print(f"training {n_trainable_l4:,} of {count(model):,} parameters")     # 8,395,267 of 11,178,051

#### 9.2 Fine-tuning (lr = 1e-4, 6 epochs, flips on)

In [ ]:
hist_ft, best_ft = fit(model, trainable, epochs=6, lr=1e-4)
record("layer4", "ResNet-18 fine-tuned: layer4 + fc", model, hist_ft, best_ft, n_trainable_l4, lr=1e-4)
print("best epoch", best_ft["epoch"], "val macro-F1", round(best_ft["f1"], 3))

#### 9.3 Where are the errors now?

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, predict(model, X_val), display_labels=CLASSES, cmap="Oranges");

### 10. Mini challenge (slide 29) — unfreeze more

> Fresh pretrained ResNet-18, unfreeze `layer3 + layer4 + fc`, `lr = 1e-4`, **same epochs** (6). Measure trainable parameters, seconds per epoch and best validation macro-F1. Compare with `layer4` only: worth it? **The test set stays closed.**

Everything else is identical to section 9 (same seed, same flips, same class weights, same `fit`), so the only difference is which weights are allowed to move.

In [ ]:
torch.manual_seed(0)
model_l34 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model_l34.fc = nn.Linear(512, 3)
for name, p in model_l34.named_parameters():
    p.requires_grad = name.startswith(("layer3", "layer4", "fc"))
trainable_l34 = [p for p in model_l34.parameters() if p.requires_grad]    # rebuild the optimizer with these
n_trainable_l34 = sum(p.numel() for p in trainable_l34)
print(f"training {n_trainable_l34:,} of {count(model_l34):,} parameters "
      f"(+{n_trainable_l34 - n_trainable_l4:,} vs layer4 only)")           # layer3 adds 2,099,712

In [ ]:
hist_l34, best_l34 = fit(model_l34, trainable_l34, epochs=6, lr=1e-4)
record("layer3_4", "ResNet-18 fine-tuned: layer3 + layer4 + fc", model_l34, hist_l34, best_l34,
       n_trainable_l34, lr=1e-4)
print("best epoch", best_l34["epoch"], "val macro-F1", round(best_l34["f1"], 3))

#### 10.1 Side by side

With only 749 validation lesions (18 Indeterminate), a small F1 difference can be noise. So besides the two numbers, I run a **paired bootstrap on the validation set**: resample the 749 lesions 2,000 times, recompute both models' macro-F1 on the same resample and look at the 95% interval of the difference. If the interval contains 0, the extra layer did not make a difference we can trust.

In [ ]:
def paired_bootstrap(y, pred_a, pred_b, n=2000, seed=0):
    """95% CI of macro-F1(b) - macro-F1(a) on the same resampled lesions, and P(b > a)."""
    rng = np.random.default_rng(seed)
    y, pred_a, pred_b = map(np.asarray, (y, pred_a, pred_b))
    diffs = np.empty(n)
    for k in range(n):
        idx = rng.integers(0, len(y), len(y))
        f = lambda p: f1_score(y[idx], p[idx], average="macro", labels=[0, 1, 2], zero_division=0)
        diffs[k] = f(pred_b) - f(pred_a)
    return np.percentile(diffs, [2.5, 97.5]), float((diffs > 0).mean())

ci, p_better = paired_bootstrap(y_val, RESULTS["layer4"]["val_pred"], RESULTS["layer3_4"]["val_pred"])
RESULTS["mini_challenge"] = {"delta_best_val_f1": round(RESULTS["layer3_4"]["best_val_f1"] - RESULTS["layer4"]["best_val_f1"], 4),
                             "delta_ci95": [round(float(c), 4) for c in ci], "p_l34_better": round(p_better, 3),
                             "time_ratio_train": round(RESULTS["layer3_4"]["sec_per_epoch_train"] / RESULTS["layer4"]["sec_per_epoch_train"], 2),
                             "params_ratio": round(n_trainable_l34 / n_trainable_l4, 3)}
save_results()

cmp = pd.DataFrame({k: {"trainable params": RESULTS[k]["trainable_params"],
                        "sec / epoch (train)": RESULTS[k]["sec_per_epoch_train"],
                        "sec / epoch (train + val)": RESULTS[k]["sec_per_epoch_total"],
                        "best val macro-F1": RESULTS[k]["best_val_f1"],
                        "best epoch": RESULTS[k]["best_epoch"],
                        "val F1 per epoch": RESULTS[k]["val_f1_per_epoch"]}
                    for k in ("layer4", "layer3_4")}).T
cmp.to_csv(OUT / "mini_challenge.csv")
print(RESULTS["mini_challenge"])
cmp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for k, c in [("layer4", "#2E6FBA"), ("layer3_4", "#F08200")]:
    axes[0].plot(RESULTS[k]["val_f1_per_epoch"], "o-", color=c, label=RESULTS[k]["label"].replace("ResNet-18 fine-tuned: ", ""))
axes[0].axhline(RESULTS["frozen"]["best_val_f1"], color="grey", ls=":", label="frozen + linear head")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("validation macro-F1"); axes[0].legend(); axes[0].set_title("validation macro-F1 per epoch")
ConfusionMatrixDisplay.from_predictions(y_val, RESULTS["layer4"]["val_pred"], display_labels=CLASSES, cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title(f"layer4 + fc (val F1 {RESULTS['layer4']['best_val_f1']:.3f})")
ConfusionMatrixDisplay.from_predictions(y_val, RESULTS["layer3_4"]["val_pred"], display_labels=CLASSES, cmap="Oranges", ax=axes[2], colorbar=False)
axes[2].set_title(f"layer3 + layer4 + fc (val F1 {RESULTS['layer3_4']['best_val_f1']:.3f})")
plt.tight_layout(); fig.savefig(FIG / "session7_mini_challenge.png", dpi=150, bbox_inches="tight"); plt.show()

**Decision.** Read the table and the bootstrap interval above. The question is whether the gain in validation macro-F1 is bigger than the noise *and* worth the extra cost per epoch. The report script (`scripts/13_session7_reports.py`) writes this conclusion from the saved numbers, so the PDF always matches what this run produced.

### 11. Homework (optional, slide 30) — full fine-tuning (strategy 3)

All 11,178,051 weights train, same lr (1e-4) and same 6 epochs. Set `RUN_FULL_FT = False` to skip it if it takes too long on the CPU.

In [ ]:
RUN_FULL_FT = True

if RUN_FULL_FT:
    torch.manual_seed(0)
    model_full = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model_full.fc = nn.Linear(512, 3)
    n_trainable_full = count(model_full)                       # everything trains
    print(f"training {n_trainable_full:,} of {count(model_full):,} parameters")
    hist_full, best_full = fit(model_full, model_full.parameters(), epochs=6, lr=1e-4)
    record("full", "ResNet-18 fully fine-tuned (all layers)", model_full, hist_full, best_full,
           n_trainable_full, lr=1e-4)
    print("best epoch", best_full["epoch"], "val macro-F1", round(best_full["f1"], 3))

### 12. Homework — the test set, once, and the final scoreboard (Sessions 5–7)

All the choices are made (architectures, epochs, which layers to unfreeze), so now the test set is opened **once** for every Session 7 model. Sessions 5–6 rows come from my saved files (`outputs/session_06/scoreboard.csv`). The **best model is the one with the highest validation macro-F1**, not the highest test score: picking on test would make its test number optimistic.

In [ ]:
test_preds = {"scratch": predict(scratch, X_test),
              "layer4": predict(model, X_test),
              "layer3_4": predict(model_l34, X_test)}
with torch.no_grad():
    test_preds["frozen"] = head(F_test.to(device)).argmax(1).cpu()
if "full" in RESULTS:
    test_preds["full"] = predict(model_full, X_test)

for k, p in test_preds.items():
    RESULTS[k]["test_macro_f1"] = round(float(f1_score(y_test, p, average="macro")), 4)
    RESULTS[k]["test_accuracy"] = round(float(accuracy_score(y_test, p)), 4)
    RESULTS[k]["test_pred"] = p.tolist()
RESULTS["y_val"], RESULTS["y_test"] = y_val.tolist(), y_test.tolist()

# Sessions 5-6, from my files
prev = pd.read_csv(config.OUTPUT_DIR / "session_06" / "scoreboard.csv", index_col=0)[["val macro-F1", "test macro-F1"]]
names7 = {"scratch": "Session 7 - SimpleCNN from scratch",
          "frozen": "Session 7 - ResNet-18 frozen + linear head",
          "layer4": "Session 7 - ResNet-18 fine-tuned layer4",
          "layer3_4": "Session 7 MC - ResNet-18 fine-tuned layer3+4",
          "full": "Session 7 HW - ResNet-18 full fine-tuning"}
s7 = pd.DataFrame({names7[k]: {"val macro-F1": round(RESULTS[k]["best_val_f1"], 3),
                               "test macro-F1": round(RESULTS[k]["test_macro_f1"], 3)}
                   for k in names7 if k in RESULTS}).T
scoreboard = pd.concat([prev, s7])
baseline_test = prev.iloc[0]["test macro-F1"]                    # my Session 5 baseline (LogReg, 306 features)
scoreboard["test Δ vs my baseline"] = (scoreboard["test macro-F1"] - baseline_test).round(3)
best_name = scoreboard["val macro-F1"].idxmax()                  # chosen on VALIDATION
scoreboard["best (by val)"] = np.where(scoreboard.index == best_name, "★", "")
scoreboard.to_csv(OUT / "scoreboard.csv")
scoreboard

#### 12.1 How sure are we about the best model's test number?

Bootstrap 95% interval of the best model's **test** macro-F1 (resampling the 749 test lesions), and its confusion matrix on test.

In [ ]:
best_key = {v: k for k, v in names7.items()}.get(best_name)      # None if a Session 5-6 model is still the best
RESULTS["scoreboard_best"] = {"name": best_name, "key": best_key,
                              "val_macro_f1": float(scoreboard.loc[best_name, "val macro-F1"]),
                              "test_macro_f1": float(scoreboard.loc[best_name, "test macro-F1"])}
if best_key is not None:
    rng = np.random.default_rng(0)
    yt, pt = y_test.numpy(), np.asarray(RESULTS[best_key]["test_pred"])
    boots = [f1_score(yt[i], pt[i], average="macro", labels=[0, 1, 2], zero_division=0)
             for i in (rng.integers(0, len(yt), len(yt)) for _ in range(2000))]
    RESULTS["scoreboard_best"]["test_ci95"] = [round(float(v), 4) for v in np.percentile(boots, [2.5, 97.5])]
    RESULTS["scoreboard_best"]["test_confusion"] = confusion_matrix(yt, pt, labels=[0, 1, 2]).tolist()
    print(best_name, RESULTS["scoreboard_best"])
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    ConfusionMatrixDisplay.from_predictions(yt, pt, display_labels=CLASSES, cmap="Oranges", ax=ax, colorbar=False)
    ax.set_title(f"{RESULTS[best_key]['label']}\nTEST (749 lesions)", fontsize=9)
    fig.savefig(FIG / "session7_best_test_confusion.png", dpi=150, bbox_inches="tight"); plt.show()
save_results()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
sb = scoreboard.iloc[::-1]
yy = np.arange(len(sb))
ax.barh(yy + 0.2, sb["val macro-F1"], height=0.4, color="#2E6FBA", label="validation")
ax.barh(yy - 0.2, sb["test macro-F1"], height=0.4, color="#F08200", label="test")
ax.set_yticks(yy); ax.set_yticklabels(sb.index, fontsize=8)
ax.axvline(baseline_test, color="k", ls=":", lw=1, label=f"my Session 5 baseline, test ({baseline_test})")
for y_, v, t in zip(yy, sb["val macro-F1"], sb["test macro-F1"]):
    ax.text(v + 0.003, y_ + 0.2, f"{v:.3f}", va="center", fontsize=7)
    ax.text(t + 0.003, y_ - 0.2, f"{t:.3f}", va="center", fontsize=7)
ax.set_xlim(0.3, max(0.8, scoreboard[["val macro-F1", "test macro-F1"]].values.max() + 0.05))
ax.set_xlabel("macro-F1"); ax.legend(loc="lower right", fontsize=8); ax.set_title("Scoreboard: Sessions 5-7, same split, same metric")
plt.tight_layout(); fig.savefig(FIG / "session7_scoreboard.png", dpi=150, bbox_inches="tight"); plt.show()

### 13. Homework — reading: EfficientNet and DenseNet vs. ResNet

* **DenseNet** (Huang et al., 2017): instead of *adding* a block's input to its output like ResNet (`F(x) + x`), each layer inside a dense block receives the **concatenation** of the feature maps of all the layers before it, so features are reused rather than recomputed, which gives strong accuracy with fewer parameters.
* **EfficientNet** (Tan & Le, 2019): instead of making the network only deeper like ResNet-18 → 152, it scales **depth, width and input resolution together** with one fixed ratio (compound scaling), on top of a small base network (found by architecture search) built from inverted-residual MBConv blocks with depthwise convolutions and squeeze-and-excitation.

### 14. Wrap-up

`outputs/session_07/results.json`, `mini_challenge.csv`, `scoreboard.csv` and the `figures/session7_*.png` are now saved. Next: `python scripts/13_session7_reports.py` → `reports/S7_MiniChallenge_Rios_Saucedo.pdf` and `reports/S7_Homework_Rios_Saucedo.pdf`.